# EDA - Dataset limpo UFC

Objetivo: olhar o dataset final salvo em `ufc-master_more_clean.csv`, entender quantos dados sobraram e listar quais features estao disponiveis para a proxima etapa de EDA/modelagem.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

csv_path = Path('ufc-master_more_clean.csv')
df = pd.read_csv(csv_path, parse_dates=['date'])

print(f'Dataset: {csv_path.name}')
print(f'Linhas: {df.shape[0]:,}')
print(f'Colunas: {df.shape[1]:,}')

df.head()

Dataset: ufc-master_more_clean.csv
Linhas: 2,625
Colunas: 34


,date,R_fighter,B_fighter,Winner,weight_class,gender,no_of_rounds,win_streak_dif,longest_win_streak_dif,lose_streak_dif,win_dif,loss_dif,total_round_dif,total_title_bout_dif,ko_dif,sub_dif,height_dif,reach_dif,age_dif,sig_str_dif,avg_sub_att_dif,avg_td_dif,B_wins,B_losses,B_avg_SIG_STR_pct,B_avg_TD_pct,B_Stance,R_wins,R_losses,R_avg_SIG_STR_pct,R_avg_TD_pct,R_Stance,R_ufc_fights,B_ufc_fights
0,2026-03-28,Israel Adesanya,Joe Pyfer,Blue,Middleweight,MALE,5,3,-5,-3,-6,-3,-48,-12,-1,2,-5.08,-12.70,-7,-0.51,0.8,1.40,7,2,0.44,0.30,Orthodox,13,5,0.48,0.09,Switch,18,9
1,2026-03-28,Michael Chiesa,Niko Price,Red,Welterweight,MALE,3,-3,-2,3,-6,3,-7,-1,4,-6,-2.54,2.54,-2,3.09,-0.4,-2.05,8,10,0.43,0.30,Orthodox,14,7,0.40,0.47,Southpaw,21,18
2,2026-03-28,Mansur Abdul-Malik,Yousri Belgaroui,Blue,Middleweight,MALE,3,-2,-1,0,-2,1,0,0,-1,-1,10.16,-2.54,5,2.82,-0.3,-1.36,2,1,0.64,1.00,Orthodox,4,0,0.44,0.41,Orthodox,4,3
3,2026-03-28,Terrance McKinney,Kyle Nelson,Red,Lightweight,MALE,3,1,1,-1,-2,-1,10,0,-2,-3,2.54,-5.08,3,-2.93,-1.6,-2.13,5,5,0.45,0.23,Switch,7,6,0.55,0.40,Switch,13,10
4,2026-03-28,Navajo Stirling,Bruno Lopes,Red,Light Heavyweight,MALE,3,-4,-2,1,-2,2,-4,0,0,0,-5.08,-12.70,4,-3.37,0.0,0.95,2,2,0.43,0.21,Orthodox,4,0,0.52,0.28,Orthodox,4,4


## 1. O que sobrou no dataset?

Esse CSV representa o recorte mais limpo ate agora: lutas masculinas, categorias escolhidas, periodo moderno e ambos os lutadores com pelo menos 2 lutas previas no UFC.

In [ ]:
resumo_dataset = pd.DataFrame({
    'metrica': [
        'linhas',
        'colunas',
        'data_inicial',
        'data_final',
        'categorias_de_peso',
        'lutadores_unicos',
    ],
    'valor': [
        len(df),
        df.shape[1],
        df['date'].min().date(),
        df['date'].max().date(),
        df['weight_class'].nunique(),
        pd.concat([df['R_fighter'], df['B_fighter']]).nunique(),
    ]
})

resumo_dataset

## 2. Features disponiveis

Abaixo esta a lista completa das colunas que sobraram, com indice para facilitar referencia durante a EDA.

In [ ]:
features = pd.DataFrame({
    'idx': range(len(df.columns)),
    'feature': df.columns,
    'tipo': df.dtypes.astype(str).values,
    'qtd_faltante': df.isna().sum().values,
    'pct_faltante': (df.isna().mean().values * 100).round(2),
})

features

## 3. Features por grupo

Separando as colunas por papel ajuda a decidir o que pode entrar como variavel explicativa e o que deve ficar fora do modelo.

In [ ]:
colunas_identificacao = ['date', 'R_fighter', 'B_fighter']
target = ['Winner']
colunas_contexto = ['weight_class', 'gender', 'no_of_rounds']
colunas_diff = [col for col in df.columns if col.endswith('_dif')]
colunas_blue = [col for col in df.columns if col.startswith('B_')]
colunas_red = [col for col in df.columns if col.startswith('R_')]
colunas_experiencia = ['R_ufc_fights', 'B_ufc_fights']

grupos_features = {
    'identificacao': colunas_identificacao,
    'target': target,
    'contexto_da_luta': colunas_contexto,
    'diferencas_pre_luta': colunas_diff,
    'blue_corner': colunas_blue,
    'red_corner': colunas_red,
    'experiencia_ufc': colunas_experiencia,
}

for grupo, colunas in grupos_features.items():
    print(f'\n{grupo} ({len(colunas)} colunas)')
    display(pd.DataFrame({'feature': colunas}))

## 4. Recortes rapidos

Conferencias basicas para entender a distribuicao do dataset final.

In [ ]:
recortes = {
    'vencedor_red_blue': df['Winner'].value_counts(dropna=False),
    'genero': df['gender'].value_counts(dropna=False),
    'categorias': df['weight_class'].value_counts(dropna=False),
    'rounds_previstos': df['no_of_rounds'].value_counts(dropna=False).sort_index(),
    'stance_red': df['R_Stance'].value_counts(dropna=False),
    'stance_blue': df['B_Stance'].value_counts(dropna=False),
}

for nome, serie in recortes.items():
    print(f'\n{nome}')
    display(serie)

## 5. Faltantes

Aqui vemos se ainda existem buracos depois dos filtros.

In [ ]:
faltantes = (
    df.isna().sum()
    .sort_values(ascending=False)
    .rename('qtd_faltante')
    .to_frame()
)
faltantes['pct_faltante'] = (faltantes['qtd_faltante'] / len(df) * 100).round(2)

faltantes.query('qtd_faltante > 0')